### Project 2 — CFPB Triage Latency Prediction
### Problem Definition + Target Construction & Audit

### Objective

- Define and audit the regression target for predicting the operational
triage latency of a CFPB consumer complaint.

### Core business question

- > At the time a complaint is received, how many days are expected to
- > pass before the complaint is sent to the company?

### Important principle

- The target is constructed from `Date received` and `Date sent to company`.

- `Date sent to company` is therefore used to create the target, but it must
never be used as an input feature at prediction time.

#### Imports & Paths

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent.parent

DATA_PATH = PROJECT_ROOT / "Data" / "raw" / "complaints-cfpb-raw.csv"
REPORT_DIR = PROJECT_ROOT / "Reports" / "project2"

REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data path:", DATA_PATH)

Project root: d:\GURU_PROJECTS\ML-FINTECH&BANK
Data path: d:\GURU_PROJECTS\ML-FINTECH&BANK\Data\raw\complaints-cfpb-raw.csv


In [2]:
### Load Raw data
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (81946, 16)

Columns:
['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue', 'Consumer complaint narrative', 'Company public response', 'Company', 'State', 'ZIP code', 'Tags', 'Submitted via', 'Date sent to company', 'Company response to consumer', 'Timely response?', 'Complaint ID']


#### 1. Problem Definition

#### Business Problem

- A complaint enters the CFPB complaint workflow and must eventually be
sent to the relevant company.

- The organization wants to understand and potentially predict the time
required for this operational handoff.

### Business Decision

- A prediction could eventually support:

- operational workload planning
- prioritization of potentially delayed complaints
- capacity planning
- SLA monitoring
- identification of operational bottlenecks

### ML Problem:- Given information available when a complaint is received, predict the
number of days until the complaint is sent to the company.

### ML Type:- Supervised regression.

### Observation Grain:- One row = one consumer complaint.

### Prediction Event:- Complaint received / available for intake processing.

### Prediction Timestamp:- `Date received`

### Target:- `triage_delay_days`= `Date sent to company - Date received`

### Important Boundary:- This model predicts an operational process latency.

- It does NOT directly predict:

- fraud
- consumer harm
- monetary loss
- complaint severity
- company response
- final resolution outcome

In [3]:
## Required Columns Verifications
required_columns = [
    "Date received",
    "Date sent to company",
    "Complaint ID"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("All required columns are present.")

All required columns are present.


#### 2. Target Construction

The target represents the elapsed operational time between:

1. Complaint received
2. Complaint sent to company

Target definition:

`triage_delay_days = Date sent to company - Date received`

This is a derived target, not an original CFPB column.

### Why this matters

The model will eventually learn:

`Available intake information → Expected operational delay`

The actual `Date sent to company` is observed only after the process
occurs, so it cannot become a model feature.

In [4]:
### Parse dates
df["Date received"] = pd.to_datetime(
    df["Date received"],
    errors="coerce",
    utc=True
)

df["Date sent to company"] = pd.to_datetime(
    df["Date sent to company"],
    errors="coerce",
    utc=True
)

print("Date received missing:", df["Date received"].isna().sum())
print("Date sent missing:", df["Date sent to company"].isna().sum())

Date received missing: 0
Date sent missing: 0


In [5]:
## Construct the Target
df["triage_delay_days"] = (
    df["Date sent to company"] - df["Date received"]
).dt.total_seconds() / (24 * 60 * 60)

print(df["triage_delay_days"].describe())

count    81946.000000
mean         1.744687
std          8.058747
min          0.000208
25%          0.005116
50%          0.009907
75%          0.019282
max        146.791586
Name: triage_delay_days, dtype: float64


#### 3. Target Audit

Before modeling, verify that the target is:

- present
- numerically valid
- non-negative
- finite
- logically consistent with the source timestamps
- sufficiently variable to justify regression

We do NOT remove extreme values at this stage.

- A long delay may represent a genuine operational event rather than noise.

In [ ]:
## Target quality Checks
target = df["triage_delay_days"]

audit = {
    "row_count": len(df),
    "target_missing": int(target.isna().sum()),
    "target_zero": int((target == 0).sum()),
    "target_negative": int((target < 0).sum()),
    "target_positive": int((target > 0).sum()),
    "target_infinite": int(np.isinf(target).sum()),
    "target_unique": int(target.nunique()),
}

audit_df = pd.DataFrame(
    audit.items(),
    columns=["check", "value"]
)

display(audit_df) # without use the display() fun same results

,check,value
0,row_count,81946
1,target_missing,0
2,target_zero,0
3,target_negative,0
4,target_positive,81946
5,target_infinite,0
6,target_unique,10806


In [7]:
## Identify the Invalid Rows
invalid_target_mask = (
    target.isna()
    | ~np.isfinite(target)
    | (target < 0)
)

invalid_rows = df.loc[
    invalid_target_mask,
    [
        "Complaint ID",
        "Date received",
        "Date sent to company",
        "triage_delay_days"
    ]
]

print("Invalid target rows:", len(invalid_rows))

if len(invalid_rows) > 0:
    display(invalid_rows.head(20))

Invalid target rows: 0


In [8]:
## Timestamp Consistency
timestamp_consistency = pd.DataFrame({
    "date_received": [
        df["Date received"].min(),
        df["Date received"].max()
    ],
    "date_sent_to_company": [
        df["Date sent to company"].min(),
        df["Date sent to company"].max()
    ]
}, index=["min", "max"])

display(timestamp_consistency)

,date_received,date_sent_to_company
min,2026-03-01 00:41:56+00:00,2026-03-01 04:58:34+00:00
max,2026-08-11 14:27:33+00:00,2026-08-27 18:14:56+00:00


#### Target Distribution Audit:
- What kind of target have we created?

In [9]:
target_summary = target.describe(
    percentiles=[
        0.01,
        0.05,
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99
    ]
)

display(target_summary)

count    81946.000000
mean         1.744687
std          8.058747
min          0.000208
1%           0.000278
5%           0.001678
10%          0.002558
25%          0.005116
50%          0.009907
75%          0.019282
90%          0.041053
95%         10.914462
99%         43.810261
max        146.791586
Name: triage_delay_days, dtype: float64

In [10]:
## Key Tail Statastics
positive_target = target[target > 0]

tail_summary = pd.Series({
    "mean_days": target.mean(),
    "median_days": target.median(),
    "p75_days": target.quantile(0.75),
    "p90_days": target.quantile(0.90),
    "p95_days": target.quantile(0.95),
    "p99_days": target.quantile(0.99),
    "max_days": target.max(),
    "zero_day_count": int((target == 0).sum()),
    "positive_day_count": int((target > 0).sum()),
})

display(tail_summary.to_frame("value"))

,value
mean_days,1.744687
median_days,0.009907
p75_days,0.019282
p90_days,0.041053
p95_days,10.914462
p99_days,43.810261
max_days,146.791586
zero_day_count,0.000000
positive_day_count,81946.000000


In [11]:
## Extreme Target Investigation
extreme_threshold = target.quantile(0.99)

extreme_rows = (
    df.loc[
        target >= extreme_threshold,
        [
            "Complaint ID",
            "Date received",
            "Date sent to company",
            "Product",
            "Company",
            "triage_delay_days"
        ]
    ]
    .sort_values("triage_delay_days", ascending=False)
)

print("99th percentile threshold:", extreme_threshold)
print("Rows at/above threshold:", len(extreme_rows))

display(extreme_rows.head(20))


99th percentile threshold: 43.81026099537039
Rows at/above threshold: 820


,Complaint ID,Date received,Date sent to company,Product,Company,triage_delay_days
14361,20379834,2026-03-18 17:19:32+00:00,2026-08-12 12:19:25+00:00,Debt collection,"Hayt Hayt & Landau, P.L. (FL)",146.791586
21102,19987390,2026-03-05 02:00:50+00:00,2026-07-22 18:22:58+00:00,Debt collection,"CITIBANK, N.A.",139.682037
9303,20273964,2026-03-14 22:13:44+00:00,2026-07-30 20:34:42+00:00,Debt collection,"MRS BPO, LLC",137.931227
14222,20737733,2026-03-28 00:54:21+00:00,2026-08-12 20:27:22+00:00,Checking or savings account,Albert Corporation,137.814595
78387,20420820,2026-03-19 18:53:27+00:00,2026-07-30 20:24:03+00:00,Debt collection,"FAIR COLLECTIONS & OUTSOURCING, INC.",133.062917
23714,20021840,2026-03-05 23:56:01+00:00,2026-07-14 15:17:53+00:00,Debt collection,"I.C. System, Inc.",130.640185
21825,20091651,2026-03-09 14:39:30+00:00,2026-07-07 20:15:29+00:00,Mortgage,"JHL, LLC",120.233322
81553,19949603,2026-03-04 00:29:05+00:00,2026-07-01 13:04:09+00:00,Debt collection,Concord Servicing Corporation,119.524352
45895,20418771,2026-03-19 17:51:22+00:00,2026-07-16 17:52:48+00:00,Mortgage,Liberty 1 Lending Inc.,119.000995
74095,19980225,2026-03-04 21:37:32+00:00,2026-07-01 13:13:30+00:00,Debt collection,Self Financial Inc.,118.649977


## Insights: What we're looking for

- Not:- "These values are large, so remove them."

- Instead:- "Are these extreme delays genuine operational observations?"

- That distinction matters enormously in operational regression.

#### 4. Target Integrity Decision

### Target acceptance criteria

The target is considered structurally valid if:

- required timestamps exist
- target can be deterministically reproduced
- no negative durations exist
- values are finite
- the target contains meaningful variation
- extreme values are explainable as possible real operational delays

### Important decision:- Do NOT remove extreme delays simply because they hurt regression metrics.

- They may represent the exact operational problem the organization wants
to understand.

- Any later transformation, such as `log1p(triage_delay_days)`, will be
considered only after the dedicated Target EDA phase.

In [12]:
## Automation Decision Summary
target_valid = (
    target.notna().all()
    and np.isfinite(target).all()
    and (target >= 0).all()
    and target.nunique() > 1
)

decision = {
    "target_name": "triage_delay_days",
    "target_valid_structurally": bool(target_valid),
    "negative_values": int((target < 0).sum()),
    "missing_values": int(target.isna().sum()),
    "infinite_values": int(np.isinf(target).sum()),
    "unique_values": int(target.nunique()),
    "mean_days": float(target.mean()),
    "median_days": float(target.median()),
    "max_days": float(target.max()),
}

display(pd.DataFrame(
    decision.items(),
    columns=["metric", "value"]
))

,metric,value
0,target_name,triage_delay_days
1,target_valid_structurally,True
2,negative_values,0
3,missing_values,0
4,infinite_values,0
5,unique_values,10806
6,mean_days,1.744687
7,median_days,0.009907
8,max_days,146.791586


### Save Audited Dataset

- One important distinction: We are not creating the final ML-ready dataset. Instead We're saving the target-audited working data so the next notebook can reproduce the analysis.

In [13]:
audit_output_path = REPORT_DIR / "project2_target_audit_summary.json"

with open(audit_output_path, "w") as f:
    json.dump(decision, f, indent=4)

print("Saved:", audit_output_path)

Saved: d:\GURU_PROJECTS\ML-FINTECH&BANK\Reports\project2\project2_target_audit_summary.json


#### PROJECT 2 — ANALYSIS INSIGHTS:- Problem Definition + Target Audit
============================================================

1. Prediction objective:
   Predict complaint triage latency from information available
   at complaint intake.

2. Prediction timestamp:
   Date received.

3. Target:
   triage_delay_days =
   Date sent to company - Date received.

4. Date sent to company is target-construction information,
   NOT a permitted model feature.

5. The target represents operational process latency rather than
   customer outcome or complaint severity.

6. Target quality must be established before feature engineering
   or model training.

7. Extreme delays are not automatically treated as outliers.
   They may represent genuine operational failures or bottlenecks.

8. Target distribution characteristics will determine whether
   raw regression, log-transformed regression, or another modeling
   strategy is appropriate.

9. The next phase is Prediction-Time Leakage Audit.
   Only information legitimately available at Date received
   should survive that audit as candidate predictors.

10. No ML model should be trained before the target and
    prediction-time information boundary are frozen.
